# Graph Zoo Designer
Build, inspect, and save a graph zoo for a simulation run.

**Workflow:**
1. Run the cells for the graph types you want (comment out the rest)
2. Visualize with `zoo.draw_all()`
3. Set a batch name and save
4. Pass the zoo to `ProcessLab`

In [ ]:
from moran_process import GraphZoo, PopulationGraph

## Section 1 — Build the zoo
Run the cells for the graphs you want. Skip the rest.

In [ ]:
BATCH_NAME = "2026-05-19_large_batch_test_ml"   # <-- change this for each experiment 
zoo = GraphZoo(name=BATCH_NAME)

In [ ]:
# Mammalian lung (balanced binary tree)
zoo.add(PopulationGraph.mammalian_lung_graph(branching_factor=2, depth=4))

In [ ]:
# Avian lung (parallel rods with inlet/outlet)
zoo.add(PopulationGraph.avian_graph(n_rods=4, rod_length=7))
zoo.add(PopulationGraph.avian_graph(n_rods=7, rod_length=4))


In [ ]:
# Fish gill (comb structure)
zoo.add(PopulationGraph.fish_graph(n_rods=3, rod_length=3))

In [ ]:
# Complete graph (well-mixed baseline)
zoo.add(PopulationGraph.complete_graph(n_nodes=31))

In [ ]:
# Cycle graph
zoo.add(PopulationGraph.cycle_graph(n_nodes=31))

In [ ]:
# Random connected graphs (null model) — multiple seeds for statistics
for seed in range(1000):
    zoo.add(PopulationGraph.random_connected_graph(n_nodes=31, n_edges=34, seed=seed))

In [ ]:
# Summary
print(zoo)
print(f"Total graphs: {len(zoo)}")

## Section 2 — Visualize

In [ ]:
%matplotlib inline
zoo.draw_all(cols=3)

## Section 3 — Save

In [ ]:
zoo_path = f"../simulation_data/{BATCH_NAME}/zoo.pkl"
zoo.save(zoo_path)

## Section 4 — Run simulation
Pass the saved zoo to `ProcessLab`. Use `run_comparative_study` for local runs, `submit_jobs` for HPC.

In [ ]:
from moran_process import GraphZoo, ProcessLab

zoo = GraphZoo.load(zoo_path)
lab = ProcessLab()

# --- Local (small scale) ---
# df = lab.run_comparative_study(
#     graphs_zoo=list(zoo),
#     r_values=[1.0, 1.1, 1.2, 2.0],
#     n_repeats=100,
#     output_path=f"../simulation_data/tmp/batch_{BATCH_NAME}/results_local.csv"
# )

# --- HPC (full scale) ---
lab.submit_jobs(
    # graphs_zoo=list(zoo),
    zoo_path=zoo_path,
    r_values=[1.1],
    n_repeats=10_000,
    n_requested_jobs=1000,
    n_graphs=len(zoo),
    queue="gsla-cpu",
    batch_dir=f"../simulation_data/{BATCH_NAME}",
    batch_name=BATCH_NAME
)